# Feature Engineering for Spotify Listening History Data
---

**Goal:**
- Create higher-level behavioral features 
- Prepare for insight-driven visuals

**Tasks:**
1. Load datasets
2. Time-Based Feature Engineering
3. Engagement Features
4. Listening Session Id's
5. Session-Level Features

In [12]:
#Libraries
import pandas as pd
import numpy as np
import os

## Load
---

In [13]:
#Load cleaned datasets
df_a = pd.read_csv("../Cleaned Data/User_A_FE_Ready.csv", parse_dates=['ts'], low_memory=False)
df_b = pd.read_csv("../Cleaned Data/User_B_FE_Ready.csv", parse_dates=['ts'], low_memory=False)

#Quick sanity check
for name, df in zip(['User A', 'User B'], [df_a, df_b]):
    print(name, "shape:", df.shape)
    print(name, "time range:", df['ts'].min(), "to", df['ts'].max())
    print(name, "missing values (top 5):\n", df.isnull().sum().sort_values(ascending=False).head())
    print("-"*50)

User A shape: (136382, 13)
User A time range: 2021-11-03 18:04:27+00:00 to 2025-08-04 19:57:08+00:00
User A missing values (top 5):
 offline_timestamp                    7343
ts                                      0
ms_played                               0
master_metadata_track_name              0
master_metadata_album_artist_name       0
dtype: int64
--------------------------------------------------
User B shape: (60357, 13)
User B time range: 2021-11-03 18:15:30+00:00 to 2025-08-04 19:59:02+00:00
User B missing values (top 5):
 offline_timestamp                    34005
ts                                       0
ms_played                                0
master_metadata_track_name               0
master_metadata_album_artist_name        0
dtype: int64
--------------------------------------------------


## Feature Engineering
---
New Features Created:
1. Temporal Features
   - `year`, `month`, `day`
   - `day_of_week` (0: Mon - 6: Sun)
   - `hour` (0-23 hr)
   - `is_weekend` (Sat or Sun)
2. Playback Metrics
   - `min_played` (ms_played to min)
   - `total_min` (total mins listened per day)
   - `track_count` (number of tracks played per day)
   - `avg_track_min` (avg track duration per day)
   - `shuffle_pct`, `skipped_pct`, `offline_pct` (% per day)
3. Top Tracks
   - `track_play_count`

### Basic Time-Features
---

In [14]:
#Extract: year, month, day, day_of_week, hour, is_weekend (sat/sun)
#Monday=0, Sunday=6
def add_time_features(df):
    df['year'] = df['ts'].dt.year
    df['month'] = df['ts'].dt.month
    df['day'] = df['ts'].dt.day
    df['day_of_week'] = df['ts'].dt.dayofweek
    df['hour'] = df['ts'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5,6])
    return df

df_a = add_time_features(df_a)
df_b = add_time_features(df_b)

#Check
df_a.head(3), df_b.head(3)

(                         ts  ms_played  \
 0 2021-11-03 18:04:27+00:00       4840   
 1 2021-11-03 18:04:33+00:00       5530   
 2 2021-11-03 18:04:40+00:00       6510   
 
                  master_metadata_track_name master_metadata_album_artist_name  \
 0  Wish Me Luck (Extended Explicit Version)                           50 Cent   
 1                                In My Hood                           50 Cent   
 2                                This Is 50                           50 Cent   
 
            master_metadata_album_album_name  \
 0  Wish Me Luck (Extended Explicit Version)   
 1                              The Massacre   
 2                              The Massacre   
 
                       spotify_track_uri reason_start reason_end  shuffle  \
 0  spotify:track:721LjayaMLcB0GFqUedvUW      playbtn    endplay    False   
 1  spotify:track:3X5tOT7Aif4l4ZjRj217pa     clickrow    endplay    False   
 2  spotify:track:0u8iiPBZ5OqK1mbq1cZc0c     clickrow    endplay    Fal

## Listening Duration Features
---

In [15]:
#min_played = ms_played / 60000
for df in [df_a, df_b]:
    df['min_played'] = df['ms_played'] / 60000

#Quick check
print(df_a[['ms_played','min_played']].head())
print(df_b[['ms_played','min_played']].head())

   ms_played  min_played
0       4840    0.080667
1       5530    0.092167
2       6510    0.108500
3      19430    0.323833
4       8700    0.145000
   ms_played  min_played
0      72358    1.205967
1      39381    0.656350
2       6378    0.106300
3      26304    0.438400
4       2581    0.043017


### Track/ Artists & Album Play Count
---

In [16]:
#Count of track, artist & album
for df in [df_a, df_b]:
    df['track_play_count'] = df.groupby('master_metadata_track_name')['master_metadata_track_name'].transform('count')
    df['artist_play_count'] = df.groupby('master_metadata_album_artist_name')['master_metadata_album_artist_name'].transform('count')
    df['album_play_count'] = df.groupby('master_metadata_album_album_name')['master_metadata_album_album_name'].transform('count')

#Quick check
df_a[['master_metadata_track_name','track_play_count']].drop_duplicates().head(3)
df_b[['master_metadata_track_name','track_play_count']].drop_duplicates().head(3)

,master_metadata_track_name,track_play_count
0,Robbery,328
1,Bandit (with YoungBoy Never Broke Again),102
2,Empty,358


### User Behavior Features (Boolean Columns)
---

In [17]:
#Ensure boolean columns are proper type (`shuffle`, `skipped`, `offline`, `incognito_mode`)
for df in [df_a, df_b]:
    df['shuffle'] = df['shuffle'].astype(bool)
    df['skipped'] = df['skipped'].astype(bool)
    df['offline'] = df['offline'].astype(bool)
    df['incognito_mode'] = df['incognito_mode'].astype(bool)

### Aggregate Listening Per Day
---

In [18]:
#Calculate daily listening stats (total mins, tracks per day, avg. length)
def daily_agg(df):
    daily = df.groupby(df['ts'].dt.date).agg(
        total_min=('min_played','sum'),
        track_count=('master_metadata_track_name','count'),
        avg_track_min=('min_played','mean'),
        shuffle_pct=('shuffle','mean'),
        skipped_pct=('skipped','mean'),
        offline_pct=('offline','mean')
    ).reset_index().rename(columns={'ts':'date'})
    return daily

daily_a = daily_agg(df_a)
daily_b = daily_agg(df_b)

daily_a.head(3), daily_b.head(3)

(         date   total_min  track_count  avg_track_min  shuffle_pct  \
 0  2021-11-03  190.721717          100       1.907217         0.55   
 1  2021-11-04  177.471550           49       3.621868         1.00   
 2  2021-11-09    2.655583            1       2.655583         1.00   
 
    skipped_pct  offline_pct  
 0          0.0          0.0  
 1          0.0          0.0  
 2          0.0          0.0  ,
          date   total_min  track_count  avg_track_min  shuffle_pct  \
 0  2021-11-03   38.894583          129       0.301508     0.689922   
 1  2021-11-04  276.909067          184       1.504941     0.717391   
 2  2021-11-05  310.231783          151       2.054515     0.993377   
 
    skipped_pct  offline_pct  
 0          0.0     0.000000  
 1          0.0     0.000000  
 2          0.0     0.033113  )

In [19]:
#Save User A & User B feature-engineered datasets
df_a.to_csv("../Cleaned Data/User_A_Finalized.csv", index=False)
df_b.to_csv("../Cleaned Data/User_B_Finalized.csv", index=False)

print("Feature-engineered |Finalized datasets saved successfully!")

Feature-engineered |Finalized datasets saved successfully!
